In [33]:
from typing import TypedDict, Annotated, Literal
#from langgraph.graph import StateGraph, START, END
import fitz  # PyMuPDF
import requests
import random
import io
from PIL import Image
from langdetect import detect,LangDetectException
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM
import base64
from langchain_ollama import ChatOllama


In [34]:
llm = ChatOllama(model="glm-ocr")

In [35]:
import pytesseract

# This links your Python library to the software you just installed
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [36]:
fraction_doc=0.4 #limit to what fraction of doc should be covered with text to not pass for OCR

In [37]:
def getdoc(source:str):
    if isinstance(source,str) and source.startswith("http"):
        resp=requests.get(source, timeout=30)
        resp.raise_for_status()
        doc=fitz.open(stream=resp.content,filetype="pdf")
    else:
        doc = fitz.open(str(source))
    return doc

In [38]:
def get_text_fraction(doc, pages_to_include):
    """
    Calculates text area fraction for a list of pages in an open fitz.Document.
    :param doc: An opened fitz.Document object
    :param pages_to_include: List of zero-indexed page numbers
    :return: Fraction (float) of text area vs total page area
    """
    total_text_area = 0
    total_page_area = 0
    
    for p_no in pages_to_include:
        # Safety check: skip if page index is out of bounds
        if p_no < 0 or p_no >= doc.page_count:
            continue
            
        page = doc[p_no]
        
        # 1. Page Area (Standard PDF points)
        total_page_area += page.rect.width * page.rect.height
        
        # 2. Text Block Area 
        # get_text("blocks") returns a list of tuples: (x0, y0, x1, y1, "text", block_no, block_type)
        for b in page.get_text("blocks"):
            x0, y0, x1, y1 = b[:4]
            block_area = (x1 - x0) * (y1 - y0)
            total_text_area += block_area
            
    if total_page_area == 0:
        return 0.0
        
    return total_text_area / total_page_area

In [39]:
def is_OCR_required(source):
    num_pages = source.page_count
    sample_pages=random.sample(range(1,num_pages+1),max(3,int(num_pages*0.1)))
    fraction=get_text_fraction(source,sample_pages)
    if fraction<fraction_doc:
        return True
    else:
        return False

In [40]:
import tempfile
from pathlib import Path

import cv2
import numpy as np


def deskew_image(image_path, output_path=None):
    image_path = Path(image_path)
    output_path = Path(output_path) if output_path else image_path.with_name("deskewed_image.png")
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    img = cv2.bitwise_not(img)
    coords = np.column_stack(np.where(img > 0))
    if coords.size == 0:
        cv2.imwrite(str(output_path), cv2.bitwise_not(img))
        return str(output_path)

    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(
        cv2.bitwise_not(img),
        matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )
    cv2.imwrite(str(output_path), rotated)
    return str(output_path)


def remove_noise(image_path, output_path=None):
    image_path = Path(image_path)
    output_path = Path(output_path) if output_path else image_path.with_name("cleaned_image.png")
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    kernel = np.ones((1, 1), np.uint8)
    img = cv2.dilate(img, kernel, iterations=1)
    img = cv2.erode(img, kernel, iterations=1)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    cv2.imwrite(str(output_path), img)
    return str(output_path)


def preprocess_image(image_path):
    cleaned_path = remove_noise(image_path)
    return deskew_image(cleaned_path)

In [41]:
from langchain_core.messages import HumanMessage

def OCR(doc: fitz.Document) -> str:
    md_sections: list[str] = []

    with tempfile.TemporaryDirectory() as temp_dir:
        temp_dir_path = Path(temp_dir)

        for page_index in range(len(doc)):
            page = doc[page_index]
            image_list = page.get_images(full=True)
            page_texts: list[str] = []

            if image_list:
                for image_index, image_info in enumerate(image_list):
                    xref = image_info[0]
                    extracted_image = doc.extract_image(xref)
                    image_bytes = extracted_image["image"]
                    image_ext = extracted_image.get("ext", "png")

                    image_path = temp_dir_path / f"page_{page_index + 1}_image_{image_index + 1}.{image_ext}"
                    image_path.write_bytes(image_bytes)

                    try:
                        cleaned_path = remove_noise(
                            image_path,
                            temp_dir_path / f"page_{page_index + 1}_image_{image_index + 1}_cleaned.png",
                        )
                        deskewed_path = deskew_image(
                            cleaned_path,
                            temp_dir_path / f"page_{page_index + 1}_image_{image_index + 1}_deskewed.png",
                        )

                        with open(deskewed_path, "rb") as image_file:
                            processed_bytes = image_file.read()

                        image_base64 = base64.b64encode(processed_bytes).decode("utf-8")
                        message = HumanMessage(
                            content=[
                                {
                                    "type": "text",
                                    "text": "Extract the text from this image exactly as shown. Return only the transcribed text.",
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/png;base64,{image_base64}"},
                                },
                            ]
                        )

                        response = llm.invoke([message])
                        ocr_text = response.content.strip()

                        if ocr_text:
                            page_texts.append(f"**Image {image_index + 1}**\n\n{ocr_text}")
                    except Exception as e:
                        print(f"Error processing image {image_index + 1} on page {page_index + 1}: {e}")
            else:
                page_image_path = temp_dir_path / f"page_{page_index + 1}.png"
                pixmap = page.get_pixmap(dpi=300, alpha=False)
                pixmap.save(str(page_image_path))

                try:
                    cleaned_path = remove_noise(
                        page_image_path,
                        temp_dir_path / f"page_{page_index + 1}_cleaned.png",
                    )
                    deskewed_path = deskew_image(
                        cleaned_path,
                        temp_dir_path / f"page_{page_index + 1}_deskewed.png",
                    )

                    with open(deskewed_path, "rb") as image_file:
                        processed_bytes = image_file.read()

                    image_base64 = base64.b64encode(processed_bytes).decode("utf-8")
                    message = HumanMessage(
                        content=[
                            {
                                "type": "text",
                                "text": "Extract the text from this page image exactly as shown. Return only the transcribed text.",
                            },
                            {
                                "type": "image_url",
                                "image_url": {"url": f"data:image/png;base64,{image_base64}"},
                            },
                        ]
                    )

                    response = llm.invoke([message])
                    ocr_text = response.content.strip()

                    if ocr_text:
                        page_texts.append(f"**Image 1**\n\n{ocr_text}")
                except Exception as e:
                    print(f"Error processing page {page_index + 1}: {e}")

            if page_texts:
                md_sections.append(
                    f"## Page {page_index + 1}\n\n" + "\n\n---\n\n".join(page_texts)
                )

    if not md_sections:
        return "*No OCR text could be extracted from this document.*"

    return "\n\n".join(md_sections)

In [42]:
def extract(doc: fitz.Document) -> str:
    """
    Extract native text layers from every page of a PDF.
    (Renamed from extract_text to match the custom pipeline snippet).
    """
    md_sections: list[str] = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        raw_text = page.get_text("text").strip()

        if not raw_text:
            continue  # blank or fully image-based page

        section = f"## Page {page_index + 1}\n\n{raw_text}"
        md_sections.append(section)

    if not md_sections:
        return "*No native text layers were found in this document.*"

    return "\n\n".join(md_sections)

In [43]:
def _is_english(text: str) -> bool:
    """Return True if *text* is detected as English, False otherwise."""
    if len(text.strip()) < 20:
        return False
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False


def english_filtered_text(text: str) -> str:
    """Return only English content from a Markdown-formatted PDF string."""
    import re

    page_pattern = re.compile(r"(?=^## Page \d+)", re.MULTILINE)
    page_sections = [s.strip() for s in page_pattern.split(text) if s.strip()]

    output_sections: list[str] = []

    for section in page_sections:
        lines = section.split("\n", 1)
        if len(lines) == 2 and lines[0].startswith("## Page"):
            page_heading = lines[0].strip()
            body = lines[1].strip()
        else:
            page_heading = None
            body = section

        raw_blocks = [b.strip() for b in body.split("\n\n---\n\n") if b.strip()]
        english_blocks: list[str] = []

        for block in raw_blocks:
            block_lines = block.split("\n\n", 1)
            if (
                len(block_lines) == 2
                and block_lines[0].startswith("**")
                and block_lines[0].endswith("**")
            ):
                label   = block_lines[0]
                content = block_lines[1]
            else:
                label   = None
                content = block

            if _is_english(content):
                english_blocks.append(block)

        if not english_blocks:
            continue

        body_md = "\n\n---\n\n".join(english_blocks)
        if page_heading:
            output_sections.append(f"{page_heading}\n\n{body_md}")
        else:
            output_sections.append(body_md)

    return "\n\n".join(output_sections)

In [44]:
def extract_text(path: str) -> str:
    """Main pipeline combining reading, native vs OCR parsing, and language filtering."""
    doc = getdoc(path)
    if is_OCR_required(doc):
        text = OCR(doc)
    else:
        text = extract(doc)
    text = english_filtered_text(text)
    return text


In [45]:
text=extract_text("C:\\Users\\vaibh\\Downloads\\L.A.BILL 18 of 2021.pdf")

In [46]:
text

'## Page 1\n\n**Image 1**\n\nL. A. BILL No. XVIII OF 2021.\n\nA BILL\n\nto amend the Farmers\' Produce Trade and Commerce (Promotion and Facilitation) Act, 2020, in its application to the State of Maharashtra.\n\nWHEREAS it is expedient to amend the Farmers\' Produce Trade and Commerce (Promotion and Facilitation) Act, 2020, in its application to the State of Maharashtra, for the purposes hereinafter appearing; it is hereby enacted in the Seventy-second Year of the Republic of India as follows:\n\n1. (1) This Act may be called the Farmers\' Produce Trade and Commerce (Promotion and Facilitation) (Maharashtra Amendment) Act, 2021.\n\n(2) It shall extend to the whole of the State of Maharashtra.\n\n(3) It shall come into force on such date, as the State Government may, by notification in the Official Gazette, appoint.\n\n## Page 2\n\n**Image 1**\n\n2. In section 1 of the Farmers\' Produce Trade and Commerce (Promotion and Facilitation) Act, 2020, in its application to the State of Mahara